<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Measure the error that matters

**[Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/)** · Forecasting and Time Series: Predict What Comes Next · Module 3, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score a forecast with MAE, RMSE, MAPE, WAPE, MASE and bias, to see where each one breaks, and to print the first rows of the course's backtest table.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The rolling-origin backtest, from [Test like the future: backtesting](https://learning.nextia-ai.com/courses/forecasting/m03/test-like-the-future-backtesting/). MAE and RMSE, from [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/) in the machine learning course (this notebook repeats them in short). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M03-L03-measure-the-error-that-matters/measure-the-error-that-matters-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One week, six measures

A garden lamp sold `4, 2, 0, 3, 5, 8, 6` in one week and `5, 3, 1, 3, 6, 10, 7` the next (Monday to Sunday). The seasonal naive forecast for the third week repeats the second week. The third week really sold `6, 2, 0, 4, 5, 12, 9`. This is a small example that we made up, so you can check every number.

> **Predict.** The lamp sold nothing on Wednesday. Which measure will fail on that day?

In [ ]:
week1 = np.array([4, 2, 0, 3, 5, 8, 6])
week2 = np.array([5, 3, 1, 3, 6, 10, 7])
forecast = week2                         # seasonal naive: the same weekday last week
real = np.array([6, 2, 0, 4, 5, 12, 9])
error = real - forecast
print("error:", error.tolist())
with np.errstate(divide="ignore"):
    ape = np.abs(error) / real            # a share of each day's real sales
scale = np.abs(week2 - week1).mean()      # the naive miss in the training weeks
print("MAE  ", round(np.abs(error).mean(), 2))
print("RMSE ", round(np.sqrt((error ** 2).mean()), 2))
print("MAPE ", 100 * ape.mean(), "| without Wednesday:", round(100 * ape[real > 0].mean(), 1))
print("WAPE ", round(100 * np.abs(error).sum() / real.sum(), 1))
print("MASE ", round(np.abs(error).mean() / scale, 2), "| scale:", scale)
print("bias ", round(100 * (forecast - real).sum() / real.sum(), 1))

> **Check.** You should see `error: [1, -1, -1, 1, -1, 2, 2]`, then MAE `1.29`, RMSE `1.36`, MAPE `inf` (`25.1` without Wednesday), WAPE `23.7`, MASE `1.29` with scale `1.0`, and bias `-7.9`.

Each measure answers a different question:

- **MAE**: the mean miss, in lamps: 9 ÷ 7 = 1.29 a day.
- **RMSE**: the root of the mean squared miss. The two misses of 2 count four times as much as a miss of 1, so RMSE (1.36) is above MAE.
- **MAPE**: the mean miss as a share of each day's sales. On Wednesday, 1 ÷ 0 has no value, so MAPE is infinite.
- **WAPE**: all misses added up (9) as a share of all sales (38): 23.7%. A zero day is no problem.
- **MASE**: the MAE divided by the miss of the naive "same weekday last week" forecast in the training weeks (1.0). Below 1 beats that naive miss; above 1 does not.
- **bias**: forecast minus real, added up, as a share of the real sales. −7.9%: the forecast was too low.

## 2. MAPE on the 50 products

The next cell backtests the seasonal naive forecast on the 50 products and counts the forecast days with zero sales.

> **Predict.** How many of the 9,100 forecast days (50 products × 13 origins × 14 days) have zero sales?

In [ ]:
sales = load_sales(BACKTEST_PRODUCTS)
fc = bt.backtest(sales, bt.seasonal_naive(7), h=14, origins=ORIGINS)
err = fc["y"] - fc["yhat"]
with np.errstate(divide="ignore", invalid="ignore"):
    ape = err.abs() / fc["y"]
print("forecast days:", len(fc), "| with zero sales:", int((fc["y"] == 0).sum()))
print("MAPE %:", 100 * ape.mean(), "| without the zero days:", round(100 * ape[np.isfinite(ape)].mean(), 1))
print("WAPE %:", round(100 * err.abs().sum() / fc["y"].sum(), 1))

> **Check.** You should see `forecast days: 9100 | with zero sales: 824`, MAPE `inf` (`77.1` without the zero days) and WAPE `51.3`.

MAPE cannot score these forecasts. If you drop the zero days, MAPE says 77.1%: it forgets 824 days, and it gives a day with 1 sale as much weight as a day with 100. WAPE weighs each day by its sales, so it says what Kenji feels: 51.3% of the units were missed.

## 3. MAE across products

> **Predict.** The best seller of the 50 and a slow seller get the same forecast method. Which one has the larger MAE? Is it the worse forecast?

In [ ]:
row = []
for uid, g in fc.groupby("unique_id"):
    one = bt.score(g, sales, uid, season=7)
    row.append({"product": uid, "units a day": round(g["y"].mean(), 1),
                "MAE": round((g["y"] - g["yhat"]).abs().mean(), 2), "MASE": one["MASE"], "WAPE %": one["WAPE %"]})
per = pd.DataFrame(row).sort_values("units a day")
print(per.iloc[[0, -1]].to_string(index=False))

> **Check.** You should see two rows: P029 with 2.1 units a day, MAE 1.87, MASE 1.090 and WAPE 87.7; and P075 with 37.8 units a day, MAE 11.85, MASE 0.836 and WAPE 31.3.

The best seller has an MAE six times larger because it sells more, not because its forecast is worse. Its MASE and WAPE are much better. MAE has the unit of the product, so you cannot add it up or compare it across products. MASE and WAPE have no unit: MASE compares each product with its own naive miss, and WAPE with its own sales.

## 4. The first rows of the backtest table

`bt.score()` computes the columns of the course's backtest table, and `bt.show()` prints them, best MASE first. `cost $` is the cost of Kenji's 14-day order from each forecast, with the course's costs (a lost sale costs 35% of the price, a left-over unit 25%: assumptions).

> **Predict.** Which of the four baselines has the lowest cost?

In [ ]:
rows, forecasts = [], {}
for name, f in [("naive", bt.naive), ("seasonal naive", bt.seasonal_naive(7)),
                ("moving average (28 days)", bt.moving_average(28)), ("drift", bt.drift)]:
    forecasts[name] = bt.backtest(sales, f, h=14, origins=ORIGINS)
    rows.append(bt.score(forecasts[name], sales, name, season=7, cost=bt.stock_cost(PRICES)))
table = bt.show(rows)

> **Check.** You should see four rows: moving average (28 days) MASE 0.875, WAPE 43.5, cost 19444; seasonal naive 1.034, 51.3, 16739; naive 1.065, 53.2, 28261; drift 1.075, 53.8, 28570. The fit times change from run to run.

The moving average has the best MASE and WAPE. The seasonal naive has the lowest cost. Naive and drift are biased high (+7.4% and +7.8%): they repeat the last day, and the last day of each origin is a Sunday, a busy day. Every later module adds its rows to this table.

## 5. Your turn: the bias of the naive forecast

> **Your turn.** From `forecasts["naive"]` (columns `y` and `yhat`), compute the **bias %** yourself: the forecasts minus the real values, added up, divided by the real values added up, times 100. Round to 1 decimal and put it in `bias_naive`. Then run the check cell.

In [ ]:
bias_naive = ...  # the bias % of the naive forecast

In [ ]:
expect('bias_naive', bias_naive, 7.4)

A positive bias means too high: over 13 orders, the naive forecast would order 7.4% more units than Larkfield sold. A bias near zero (seasonal naive −0.5%) does not mean a good forecast: large misses up and down can cancel. Read bias next to WAPE, never alone.

## Next

You can score a forecast and you have the first four rows of the table. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Exponential smoothing (ETS, Holt-Winters)](https://learning.nextia-ai.com/courses/forecasting/m04/exponential-smoothing/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.